In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

In [2]:
train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
sample_submission = pd.read_csv('csv/sample_submission.csv')

In [3]:
#특성과 타겟 변수 분리
train = train.drop(columns=['ID'], axis = 1)
test = test.drop(columns=['ID'], axis = 1)

In [4]:
# 설립연도 타입 변환 (int -> object)
train['설립연도'] =train['설립연도'].astype('object')
test['설립연도'] =test['설립연도'].astype('object')

category_features = ['설립연도','국가','분야','투자단계','기업가치(백억원)']
numeric_features = ['직원 수','고객수(백만명)','총 투자금(억원)','연매출(억원)','SNS 팔로워 수(백만명)']
bool_features = ['인수여부','상장여부']

# LabelEncoder 객체를 각 범주형 feature별로 따로 저장하여 사용
encoders = {}

# 범주형 데이터를 encoding
for feature in category_features:
    encoders[feature] = LabelEncoder()
    train[feature] = train[feature].fillna('Missing')
    test[feature] = test[feature].fillna('Missing')
    train[feature] = encoders[feature].fit_transform(train[feature])
    test[feature] = encoders[feature].transform(test[feature])

# 불리언 값을 0과 1로 변환 ('Yes' → 1, 'No' → 0 으로 변환)
bool_map = {'Yes': 1, 'No': 0}

for feature in bool_features:
    train[feature] = train[feature].map(bool_map)
    test[feature] = test[feature].map(bool_map)

# 수치형 변수 결측치를 평균값으로 대체
for feature in numeric_features:
    mean_value = train[feature].mean()
    train[feature] = train[feature].fillna(mean_value)
    test[feature] = test[feature].fillna(mean_value)

# TabNet용 범주형 변수 인덱스(cat_idxs) 및 차원(cat_dims) 설정
features = [col for col in train.columns if col != '성공확률']
cat_idxs = [features.index(col) for col in category_features]
cat_dims = [train[col].max() + 1 for col in category_features]

/var/folders/ss/j3gw42tn6hj8yhjfpvq0shb00000gn/T/ipykernel_54400/3193102807.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[feature] = train[feature].fillna('Missing')
/var/folders/ss/j3gw42tn6hj8yhjfpvq0shb00000gn/T/ipykernel_54400/3193102807.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[feature] = test[feature].fillna('Missing')


In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

# X: features, y: soft labels (확률 값)
X_train, X_test, y_train, y_test = train_test_split(train.drop(columns='성공확률',axis=1), train['성공확률'], test_size=0.2, random_state=42)

# 회귀 모델 예시
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8
)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 확률 예측
y_pred = model.predict(X_test)

In [49]:
from sklearn.model_selection import GridSearchCV

model = GradientBoostingRegressor()

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.6, 0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,  # 5-fold 교차검증
    scoring='neg_mean_squared_error',  # 회귀라서 MSE 기준
    n_jobs=-1,  # 가능한 모든 CPU 사용
    verbose=1
)

# 학습
grid_search.fit(X_train, y_train)

# 최적의 파라미터 조합
print("Best parameters:", grid_search.best_params_)
print("Best score (MSE):", -grid_search.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best score (MSE): 0.05842315230925427


In [53]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
print(mse)

0.0581908319918075


In [54]:
print(y_pred)

[0.52873336 0.54371846 0.520719   0.5339854  0.54016301 0.53691705
 0.53375058 0.54273629 0.5326916  0.53899174 0.54251253 0.53232331
 0.54058365 0.53722807 0.53078131 0.54942209 0.53629437 0.52344964
 0.54451396 0.54487841 0.570932   0.53391924 0.54407448 0.562248
 0.54404318 0.51691569 0.53614989 0.55924091 0.54347302 0.52361397
 0.53423925 0.50611518 0.52756843 0.50319624 0.52473718 0.51839161
 0.53532259 0.54406085 0.54054414 0.54454697 0.53696305 0.54348306
 0.52211771 0.55182927 0.54644671 0.50157409 0.52931123 0.53569357
 0.52202387 0.53888425 0.53583141 0.5333346  0.52689831 0.54592166
 0.52619615 0.5166107  0.54045933 0.5236472  0.52815754 0.55358972
 0.56553991 0.52505445 0.51782751 0.53874092 0.53034273 0.56124015
 0.53454231 0.53533243 0.57956857 0.54422925 0.52759463 0.54556054
 0.53934911 0.53008142 0.52763914 0.52005003 0.56953369 0.53379394
 0.53240322 0.5622047  0.53891715 0.54009394 0.54412764 0.52490869
 0.53355703 0.52193567 0.55964997 0.49429815 0.56051339 0.502124

In [20]:
print(y_test)

2717    0.9
1969    0.6
592     0.1
939     0.6
2956    0.1
       ... 
3217    0.7
2873    0.7
2327    0.5
1904    0.9
3840    0.8
Name: 성공확률, Length: 876, dtype: float64


In [ ]:
print(y_test-y_pred)

2717    0.381770
1969    0.209666
592    -0.394658
939     0.026471
2956   -0.446946
          ...   
3217    0.182296
2873    0.182880
2327   -0.009048
1904    0.355473
3840    0.223686
Name: 성공확률, Length: 876, dtype: float64


In [51]:
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8
)
model.fit(train.drop(columns=['성공확률']), train['성공확률'])

pred = model.predict(test)

sample_submission['성공확률'] = pred
sample_submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [55]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))

Train MSE: 0.05364839053200648
Test MSE: 0.0581908319918075
